##  대화형 챗봇 (메모리)

### 🔹 Conversation Memory란?
> **Conversation Memory(대화 메모리)** 는 LLM이 **이전 대화 내용을 기억하고 이어서 대답**할 수 있도록 하는 기능입니다.

일반적인 LLM은 한 번의 요청(prompt)만 처리하고 끝나기 때문에,  
사용자가 “아까 말한 여행지 일정 다시 알려줘.”처럼 과거 대화를 참조하면 맥락을 잃어버립니다.  
→ 이때 필요한 것이 바로 **Memory(기억 기능)** 입니다.

### 🔹 왜 Memory가 필요할까?

| 상황 | Memory 없을 때 | Memory 있을 때 |
|------|----------------|----------------|
| 사용자: “부산 여행지 추천해줘.”<br>→ 모델이 답변함 | 다음 질문: “그럼 거기 근처 맛집은?”<br>→ 모델이 “어디 여행 말씀인가요?” | “부산 근처엔 회센터가 많고… 홍합탕집이 유명해요!” |
| 사용자: “어제 추천해준 책 제목 다시 말해줘.” | “어떤 책을 말씀하시는 건가요?” | “어제 말씀드린 건 『데미안』이에요.” |

👉 메모리를 활용하면,  
모델이 “과거 대화의 맥락(Context)”을 유지한 채로  
**자연스러운 멀티턴(Multi-turn) 대화형 챗봇**을 만들 수 있습니다.

### 🔹 LangChain에서의 Memory 개념

LangChain은 다양한 형태의 “기억 클래스(memory classes)”를 제공합니다.  
대표적으로 아래 세 가지를 이해하면 충분합니다 👇

| Memory 클래스 | 설명 | 특징 |
|----------------|------|------|
| **`ConversationBufferMemory`** | 단순히 대화 전체를 계속 저장 | 가장 기본적이고 직관적 |
| **`ConversationBufferWindowMemory`** | 최근 N개의 대화만 저장 | “단기 기억” 형태로, 긴 대화를 효율적으로 유지 |
| **`ConversationSummaryMemory`** | 이전 대화를 LLM이 요약해서 저장 | 긴 대화를 핵심 요약본으로 관리, 맥락 유지에 적합 |

### 🔹 프롬프트에 직접 넣는 방식 vs Memory 클래스 사용 비교

| 비교 항목 | 프롬프트에 직접 삽입 | Memory 클래스 사용 |
|------------|----------------------|---------------------|
| 코드 복잡도 | ❌ 대화마다 프롬프트를 새로 생성해야 함 | ✅ LangChain이 자동으로 이전 대화 삽입 |
| 확장성 | ❌ 대화가 길어지면 토큰 초과 문제 발생 | ✅ 오래된 대화는 요약/제한 가능 |
| 유지보수 | ⚠️ 과거 대화 삽입 위치만 교체하면 되지만, 요약·토큰 제어를 직접 구현해야 함 | ✅ 메모리 객체만 관리하면 됨 |
| 현실성 | ❌ “일회용 챗봇”에만 적합 | ✅ “지속형 대화형 챗봇” 구현 가능 |

즉, **Memory는 LLM이 “대화 히스토리를 스스로 관리”하도록 하는 스마트한 도우미 클래스**입니다.

### 🔹 Memory 작동 원리
1. 사용자가 LLM에 **질문(입력)** 을 보냅니다.  

2. LLM이 **답변**을 생성하면, 이 대화 내용이 자동으로 **Memory에 저장**됩니다.  
3. 사용자가 다음 질문을 입력하면,  
   LangChain이 **이전 대화 기록을 자동으로 프롬프트에 포함시켜** LLM에 전달합니다.  
4. LLM은 이 정보를 바탕으로 **맥락을 이해하고 연속적인 대화**를 생성합니다.

→ 즉, 사용자는 별도로 과거 대화를 넣을 필요 없이, LangChain이 **“대화 기록 → 프롬프트 삽입 → LLM 호출”**  
과정을 자동으로 처리합니다.

In [3]:
#!uv add -U langchain langchain-openai  python-dotenv tiktoken

### 코드 예시 1. 기본 Buffer Memory (대화 전체 저장)

In [6]:
### 코드 예시 ① 기본 Buffer Memory
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano")

# 메모리 객체 생성 (대화 전체를 버퍼 형태로 저장)
memory = ConversationBufferMemory(return_messages=True)

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 여행 전문가야. 사용자의 질문에 친절하게 답해줘."),
    MessagesPlaceholder(variable_name="history"),  # 이전 대화 삽입 위치
    ("human", "{input}")                           # 사용자 질문
]) 

# LCEL 체인 구성
chain = prompt | llm 

# 연속 대화 시뮬레이션
inputs = ["부산 여행지 추천해줘.", "그럼 그 근처 맛집은 어디야?"]

for user_input in inputs:
    # 메모리 불러오기
    history = memory.load_memory_variables({})["history"]
    # 실행
    result = chain.invoke({"history": history, "input": user_input})
    # 결과 출력
    print(f"\n사용자: {user_input}\n 응답: {result.content}")
    # 메모리에 저장
    memory.save_context({"input": user_input}, {"output": result.content})

C:\Users\playdata2\AppData\Local\Temp\ipykernel_4084\641154472.py:10: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(return_messages=True)



사용자: 부산 여행지 추천해줘.
 응답: 멋진 선택이에요! 부산은 바다와 도시가 만나는 매력이 있어서 취향에 맞춘 여행 코스가 정말 잘 맞아요. 아래처럼 interest별로 추천지와 간단한 포인트를 정리해볼게요.

1) 바다와 해변 분위기를 한꺼번에
- 해운대 해수욕장 + 동백섬 산책로
  - 왜 좋나요: 부산의 대표 해변. 모래사장도 길고 산책로도 멋져요. 동백섬 둘레길에서 바다 풍경을 한눈에 볼 수 있습니다.
  - 팁: 해운대 시장이나 주변 해산물 맛집에서 간단히 점심 후 산책 시작하기 좋습니다.
- 광안리 해수욕장 + 광안대교 야경
  - 왜 좋나요: 밤에 다리 조명이 반짝이고 바다와 도심이 어우러지는 멋진 야경 포인트.
  - 팁: 해질녘이나 저녁에 방문하면 더 분위기가 좋고, 광안리 해변의 카페나 포장마차도 즐길 만합니다.

2) 문화와 포토 스팟을 원한다면
- 감천문화마을
  - 왜 좋나요: 넓은 계단과 알록달록한 집들이 만드는 독특한 거리감으로 사진 찍기 좋아요.
  - 팁: 아트 갤러리나 카페도 많아 천천히 걷기 좋습니다.
- 자갈치시장 + 국제시장 + BIFF 광장(남포동 일대)
  - 왜 좋나요: 싱싱한 해산물 시장 분위기, 먹거리 골목, 영화인 거리의 감성까지 한꺼번에.
  - 팁: 해산물 회나 해물파전 같은 현지 먹거리 도전해보세요. 남포동과 광복로의 쇼핑도 함께 가능.

3) 가족·아이들과 함께라면
- 부산 아쿠아리움(해운대)
  - 왜 좋나요: 큰 규모의 해양생물 전시로 아이들도 좋아합니다.
  - 팁: 해운대에서의 일정에 맞춰 물놀이와 조개체험 같은 체험형 옵션도 체크해보세요.
- 태종대 공원
  - 왜 좋나요: 바다 절벽과 산책로, 등대 전망으로 아이들과 함께 자연 체험하기 좋습니다.
  - 팁: 산책로는 편한 신발이 좋고, 날씨가 맑을 때 더 선명한 경치를 즐길 수 있습니다.

4) 도심에서의 시티뷰와 맛집 정복
- 용두산 공원 + 부산타워
  - 왜 좋나요: 도심 전망과 함께 도심 속 숨은 맛집 찾기 좋습니다.
- 달맞이길(

모델은 앞선 “부산 여행” 대화를 기억하고 자연스럽게 이어서 대답합니다.

In [7]:
memory.load_memory_variables({})['history']

[HumanMessage(content='부산 여행지 추천해줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='멋진 선택이에요! 부산은 바다와 도시가 만나는 매력이 있어서 취향에 맞춘 여행 코스가 정말 잘 맞아요. 아래처럼 interest별로 추천지와 간단한 포인트를 정리해볼게요.\n\n1) 바다와 해변 분위기를 한꺼번에\n- 해운대 해수욕장 + 동백섬 산책로\n  - 왜 좋나요: 부산의 대표 해변. 모래사장도 길고 산책로도 멋져요. 동백섬 둘레길에서 바다 풍경을 한눈에 볼 수 있습니다.\n  - 팁: 해운대 시장이나 주변 해산물 맛집에서 간단히 점심 후 산책 시작하기 좋습니다.\n- 광안리 해수욕장 + 광안대교 야경\n  - 왜 좋나요: 밤에 다리 조명이 반짝이고 바다와 도심이 어우러지는 멋진 야경 포인트.\n  - 팁: 해질녘이나 저녁에 방문하면 더 분위기가 좋고, 광안리 해변의 카페나 포장마차도 즐길 만합니다.\n\n2) 문화와 포토 스팟을 원한다면\n- 감천문화마을\n  - 왜 좋나요: 넓은 계단과 알록달록한 집들이 만드는 독특한 거리감으로 사진 찍기 좋아요.\n  - 팁: 아트 갤러리나 카페도 많아 천천히 걷기 좋습니다.\n- 자갈치시장 + 국제시장 + BIFF 광장(남포동 일대)\n  - 왜 좋나요: 싱싱한 해산물 시장 분위기, 먹거리 골목, 영화인 거리의 감성까지 한꺼번에.\n  - 팁: 해산물 회나 해물파전 같은 현지 먹거리 도전해보세요. 남포동과 광복로의 쇼핑도 함께 가능.\n\n3) 가족·아이들과 함께라면\n- 부산 아쿠아리움(해운대)\n  - 왜 좋나요: 큰 규모의 해양생물 전시로 아이들도 좋아합니다.\n  - 팁: 해운대에서의 일정에 맞춰 물놀이와 조개체험 같은 체험형 옵션도 체크해보세요.\n- 태종대 공원\n  - 왜 좋나요: 바다 절벽과 산책로, 등대 전망으로 아이들과 함께 자연 체험하기 좋습니다.\n  - 팁: 산책로는 편한 신발이 좋고, 날씨가 맑을 때 더 선

### 코드 예시 2. 최근 대화만 유지 (Window Memory)

In [8]:
### 코드 예시 ① 기본 Buffer Memory
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano")

# 메모리 객체 생성 (대화 전체를 버퍼 형태로 저장)
memory = ConversationBufferWindowMemory(k=2, return_messages=True)

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 여행 전문가야. 사용자의 질문에 친절하게 답해줘."),
    MessagesPlaceholder(variable_name="history"),  # 이전 대화 삽입 위치
    ("human", "{input}")                           # 사용자 질문
]) 

# LCEL 체인 구성
chain = prompt | llm 

# 5️⃣ 연속 대화 시뮬레이션
inputs = ["부산 여행지 추천해줘.", "대한 민국의 수도는 어디야?", "서울의 인구는 몇명이야?", "내가 아까 추천해달라고 한 어행지는 어디야?" ]

for user_input in inputs:
    # 메모리 불러오기
    history = memory.load_memory_variables({})["history"]
    # 실행
    result = chain.invoke({"history": history, "input": user_input})
    # 결과 출력
    print(f"\n사용자: {user_input}\n 응답: {result.content}")
    # 메모리에 저장
    memory.save_context({"input": user_input}, {"output": result.content})


C:\Users\playdata2\AppData\Local\Temp\ipykernel_4084\3298101796.py:10: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(k=2, return_messages=True)



사용자: 부산 여행지 추천해줘.
 응답: 좋아요! 부산에서 꼭 가봐야 할 명소들을 분위기별로 정리해봤어. 취향이나 일정에 맞춰 골라보면 좋을 거야.

1) 해변·바다 경치가 좋은 곳
- 해운대 해수욕장: 긴 해변과 해변가 카페가 많아 산책이나 바다 풍경 감상에 좋아. 누리마루 APEC 하우스와 동백섬 산책로도 함께 추천.
- 동백섬·누리마루: 바다를 가까이에서 보고 걷기 좋은 산책로. 해가 질 때 조용하고 예뻐.
- 광안리 해수욕장 & 광안대교 야경: 밤에 다리 조명이 반짝이며 분위기 최고. 커피숍이나 루프탑 바가 많아 여유롭게 즐기기 좋음.
- 송도해수욕장 & 달맞이길: 조용한 해변 산책로와 아름다운 해안 절벽길. 봄/가을 산책에 특히 추천.
- 태종대: 해안 절벽과 등대, 파도 소리 들으며 걷기 좋은 코스. 멋진 전망대 포인트 다수.

2) 문화·시장·먹거리가 풍부한 곳
- 자갈치시장: 싱싱한 해산물 구경하고 바로 맛볼 수 있는 부산의 대표 재래시장.
- 국제시장 & BIFF 광장: 다양한 먹거리와 쇼핑, 영화제 분위기를 느낄 수 있는 곳. BIFF 광장은 특히 영화 마니아에게 간단 코스.
- 감천문화마을: 알록달록한 집과 골목 벽화가 이뤄낸 독특한 마을 풍경. 사진 찍기 좋은 곳.
- 해동용궁사: 바다를 바라보며 올려다보는 절경이 멋진 해안 사찰.
- Un 해설 없이도 재밌는 선택지: Shinsegae Centum City(신세계백화점 센텀시티)와 Spa Land에서 쇼핑+온천식 피로 풀기. 가족 단위나 커플에게 추천.

3) 바다 풍경 + 전망 포인트
- 오륙도 스카이워크: 바다 아래 투명 바닥을 걷는 독특한 체험. 사진 찍기 좋고 바다 전망이 시원해.
- 부산타워/남포동 일대 야경 산책: 낮보다 밤에 더 운치 있는 부산 도심 풍경.

4) 가족/커플/맛집 코스 팁
- 해산물 애호가라면 자갈치시장→국제시장 코스로 당일치기 추천. 신선한 해산물 구입 후 바로 조리해먹을 수 있음.
- 쇼핑과 휴식 원한다면 센텀시티(신세계)와 인근 해운대·광안리 조합이 편

이 방식은 단기 기억처럼 작동합니다.  
오래된 대화는 자동으로 지워져 토큰 낭비를 줄이고 효율성을 높입니다.

### 코드 예시 3. 이전 대화를 요약해서 저장 (ConversationSummaryMemory)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 요약형 메모리 생성 — LLM이 과거 대화를 요약해 맥락을 유지
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 여행 플래너야. 사용자의 요구에 따라 가족 여행 일정을 제안해줘."),
    MessagesPlaceholder(variable_name="history"),  # 요약된 과거 대화 자동 삽입
    ("human", "{input}")                           # 현재 질문
])

# LCEL 체인 구성
chain = prompt | llm

# 연속 대화 시뮬레이션
inputs = [
    "이번 주말에 가족 여행지 추천해줘.",
    "지난번에 추천한 곳 중에 아이들이 놀기 좋은 곳은 어디였지?",
    "그럼 거기 일정표를 하루만 짜줘."
]

for user_input in inputs:
    history = memory.load_memory_variables({})["history"]
    result = chain.invoke({"history": history, "input": user_input})
    print(f"\n👤 사용자: {user_input}\n🤖 응답: {result.content}")
    memory.save_context({"input": user_input}, {"output": result.content})

C:\Users\playdata2\AppData\Local\Temp\ipykernel_17532\2842420670.py:9: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(llm=llm, return_messages=True)



👤 사용자: 이번 주말에 가족 여행지 추천해줘.
🤖 응답: 좋아요! 이번 주말 가족 여행지로 바로 떠올릴 만한 곳을 몇 가지 스타일별로 뽑아봤어요. 다만 출발지(도시), 아이들 연령대, 예산, 차 여부, 선호 스타일을 알려주시면 더 정확하게 맞춰드릴게요.

수도권 기준으로 쉽게 다녀올 수 있는 4곳

1) 서울 근교 자연 힐링: 가평/양평
- 왜 좋나요: 아이들도 좋아하는 자연 체험과 짧은 트레킹, 물가 가까운 호수 풍경.
- 하이라이트: 남이섬 자전거, 아침고요수목원, 청평호 유람선, Petite France 등 가족 포토스팟.
- 추천 일정(2일): Day 1 - 출발 후 남이섬 자전거 + 점심, 아침고요수목원 방문, 저녁 펜션 체크인. Day 2 - 청평호 유람선/근처 체험 후 귀가.
- 포인트: 주말에는 주차/혼잡이 있을 수 있어 이른 출발이 좋고, 펜션이나 리조트 숙박이 편해요.

2) 동해안 바다 여행: 속초/강릉(1박2일)
- 왜 좋나요: 바다 풍경과 해산물 맛집, 아이들이 즐길 수 있는 짧은 산책로와 카페거리.
- 하이라이트: 속초 수산시장, 설악산 케이블카(또는 속초 해변 산책), 강릉 경포대/커피거리.
- 추천 일정: Day 1 - 출발 후 속초 도착, 수산시장 점심, 해변 산책. Day 2 - 설악산 케이블카 또는 아이들 놀거리 후 강릉으로 이동해 커피거리 방문 후 귀가.
- 포인트: 이동 시간은 교통 상황에 따라 달라지니 이른 출발 추천.

3) 테마파크 중심 당일치기/1박: 에버랜드(용인) 또는 롯데월드
- 왜 좋나요: 놀이기구를 좋아하는 아이에게 최적, 비오는 날에도 실내 공간 활용 가능.
- 하이라이트: 에버랜드의 놀이기구와 퍼레이드, 캐리비안베이 인근 수영/리조트 옵션(계절에 따라 다름).
- 추천 일정: Day 1 - 아침에 출발해 한나절 에버랜드 즐기고, 오후 또는 저녁에 숙소 체크인. Day 2 - 근처에서 가벼운 활동 뒤 귀가.
- 포인트: 주말은 혼잡하니 미리 티켓 예매, 주차 위치 확인이 중요.

4) 주말 제주도 1박

In [ ]:
# 현재까지의 요약본 확인
print(memory.buffer)

New summary: 인간은 앞서 네 곳 중 아이들이 놀기 좋을지를 묻고, AI는 출발지와 연령대, 이동 거리, 날씨를 고려해 정리하고 구체 일정 제공 여부를 안내했다. 이어서 AI는 서울·수도권 출발 기준의 ‘하루 코스 샘플’ 네 가지를 제시했고, 각 샘플은 1) 에버랜드 당일치기, 2) 롯데월드 어드벤처 당일치기, 3) 속초/강릉 당일치기, 4) 가평/양평 자연 힐링 당일치기로 구성되었다. 샘플마다 대상 아이 연령대에 맞춘 활동, 예상 소요 시간, 구체 일정, 아이 포인트와 주의점, 날씨 contingency가 포함되며, 출발지/연령/예산/차량 여부/주제/가능한 날짜를 알려주시면 즉시 맞춤 일정표와 체크리스트, 당일 주의점을 만들어 드리겠다고 안내했다. 또한 날씨 contingency를 더 구체화하는 옵션도 제시되었다.


이 방식은 LLM이 이전 대화를 스스로 요약하여 핵심 맥락을 유지합니다.  
수백 문장의 대화도 핵심 내용만 남기기 때문에 효율적입니다.

**핵심 정리**
| Memory 타입 | 특징 | 사용 목적 |
|--------------|--------|-------------|
| **BufferMemory** | 모든 대화를 그대로 저장 | 짧은 대화, 디버깅용 |
| **WindowMemory** | 최근 N개의 대화만 유지 | 실시간 채팅, 효율성 |
| **SummaryMemory** | 이전 대화를 요약해서 유지 | 긴 대화 맥락 유지형 챗봇 |

🔹 실습 : 아래 조건을 만족하는 “나만의 대화형 여행 상담 챗봇”을 만들어보세요.

당신은 AI 여행 비서 트래블GPT 입니다.  
고객이 여러 도시를 순서대로 여행하면서 맞춤 일정·숙소·음식을 요청할 때,  
이전 대화 내용을 기억하며 연결된 제안을 해주는 지능형 여행 어시스턴트를 구현하세요.  

1. ChatOpenAI(model="gpt-5-nano") 사용
2. ConversationSummaryMemory로 요약 기반 대화 기억 구현
3. 답변에 반드시 "이전 여행 내용을 바탕으로 추천드리면..." 이라는 문구 포함
  - 예시 : 이전 여행 내용을 바탕으로 추천드리면, 여수에서는 해상케이블카와 낭만포차거리를 꼭 가보세요.
4. 대화 시나리오:
  - 사용자가 “부산 → 여수 → 강릉” 순으로 도시를 이동
  - 챗봇은 이전 도시에서 한 활동을 기억하고 “연결된 여행 루트”나 “테마별 추천(가족/커플/힐링)”을 제안할 것

예시 시나리오
```bash
사용자: 이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.
AI: 부산의 해운대, 아쿠아리움이 가족 단위로 인기예요!

사용자: 이번엔 여수로 가볼까?
AI: 이전 여행 내용을 바탕으로 추천드리면, 부산의 해변 감성에 이어 여수에서는 바다 전망 케이블카와 낭만포차를 즐기세요.

사용자: 그럼 마지막은 강릉이 좋을까?
AI: 이전 여행 내용을 바탕으로 추천드리면, 강릉에서는 여수보다 조용한 힐링 카페 거리와 바다 일출 코스를 권합니다.
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 1. 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 2. 요약형 메모리 생성
memory = ConversationSummaryMemory(
    # TODO: 필요한 인자를 채워보세요.
)

# 3. 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system",
     # TODO: 트래블GPT의 역할과 반드시 포함할 문구 조건을 작성해보세요.
    ),
    # TODO: 이전 대화(history)가 들어갈 자리를 추가해보세요.
    ("human", "{input}")
])

# 4. LCEL 체인 구성
chain = ...

# 5. 대화 시나리오
inputs = [
    "이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.",
    "이번엔 여수로 가볼까?",
    "그럼 마지막은 강릉이 좋을까?"
]

# 6. 연속 대화 실행
for user_input in inputs:
    history = ...
    result = ...
    print(f"\n사용자: {user_input}\n트래블GPT: {result.content}")
    memory.save_context(...)

# 7. 선택: 요약된 메모리 확인
print(memory.buffer)

ValidationError: 1 validation error for ConversationSummaryMemory
llm
  Field required [type=missing, input_value={'human_prefix': 'Human',...'memory_key': 'history'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

<details>
<summary>정답 보기</summary>

```python
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 메모리 생성 — 이전 대화를 요약하며 맥락 유지
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 여행 비서 트래블GPT야. "
     "사용자의 여행 루트를 기억하고, 이전 여행 내용을 바탕으로 다음 도시를 추천해줘. "
     "답변에는 반드시 '이전 여행 내용을 바탕으로 추천드리면,' 이라는 문구를 포함해야 해."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# LCEL 체인 구성
chain = prompt | llm

# 대화 시나리오
inputs = [
    "이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.",
    "이번엔 여수로 가볼까?",
    "그럼 마지막은 강릉이 좋을까?"
]

# 연속 대화 시뮬레이션
for user_input in inputs:
    history = memory.load_memory_variables({})["history"]
    result = chain.invoke({"history": history, "input": user_input})
    print(f"\n사용자: {user_input}\n트래블GPT: {result.content}")
    memory.save_context({"input": user_input}, {"output": result.content})

# 대화 요약 확인 (선택)
print("\n요약된 Memory Buffer:")
print(memory.buffer)
```
</details>